# qlang · live AST and quantum schematic

One expression, one editable variable table, three live views. Run all cells from top to bottom using Python 3.14+. The setup and normal package imports follow `live_schematic.ipynb`; this notebook contains its own AST renderer and needs no AST helper loader.

The original AST and schematic notebooks are unchanged. The AST updates immediately, while circuit compilation waits for a 350 ms typing pause. If this is your first widget installation, restart the kernel after setup before continuing.

The heatmap uses the same parsed predicate and compiled oracle. Grover fits the Boolean feasibility problem; QAOA would additionally need an objective and parameter optimization. Live counts and ideal predictions are classical; actual compiled-circuit sampling runs only via **Run Grover**.


In [1]:
# Run this first. Packages are installed into the active Jupyter kernel.
from pathlib import Path
import shutil
import subprocess
import sys

if sys.version_info < (3, 14):
    raise RuntimeError("qlang requires Python 3.14 or newer. Select a compatible kernel and rerun this cell.")

requirements = [
    "qlang @ git+https://github.com/asherhe/qlang.git",
    "ipywidgets",
    "matplotlib",
]

uv = shutil.which("uv")

def install(packages):
    pip_command = [sys.executable, "-m", "pip", "install", "--upgrade", *packages]
    if uv:
        try:
            subprocess.check_call([uv, "pip", "install", "--python", sys.executable, "--upgrade", *packages])
            return
        except (OSError, subprocess.CalledProcessError):
            print("uv could not install into this kernel; falling back to pip.")
    subprocess.check_call(pip_command)

print("Installing into the active kernel:", sys.executable)
install(requirements)

# Local development uses the current checkout after the standalone GitHub install.
project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents) if (candidate / "src" / "qlang").is_dir() and (candidate / "pyproject.toml").is_file()),
    None,
)
if project_root is not None:
    install(["--editable", str(project_root)])


Installing into the active kernel: C:\Users\asher\projects\qlang\.venv\Scripts\python.exe


In [2]:
import ast
import re

import ipywidgets as widgets
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import HTML, clear_output, display
from qiskit import QuantumCircuit, QuantumRegister

import qlang


## Symbolic expression and AST

Use the **Variables** table to add/remove names, choose Boolean/UInt types, and set widths. **Full expressions** switches between expression labels and operator symbols; abbreviated labels show the full text on hover. **Show optimized graph** changes only the AST view. Colored L/R arrows identify operands, including shared children.

**Domains** accept inclusive ranges (`x = 0..7`) or lists (`x = 0, 2, 5`), one variable per line. Table changes preserve valid domains and fill in defaults for new variables. Presets restore their default variable declarations. Renaming does not rewrite the expression.


In [3]:
"""Live AST notebook UI. Reloaded from disk by the notebook launch cell."""

import ast
import html
import time
from collections import defaultdict

import ipywidgets as widgets
from IPython.display import display, HTML
import qlang
from qlang.expression import BitType, UIntType


# Parse a small expression language explicitly; never eval user text.
BINARY = {
    ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b,
    ast.Mult: lambda a, b: a * b, ast.LShift: lambda a, b: a << b,
    ast.RShift: lambda a, b: a >> b, ast.BitAnd: lambda a, b: a & b,
    ast.BitOr: lambda a, b: a | b, ast.BitXor: lambda a, b: a ^ b,
}
COMPARE = {
    ast.Eq: lambda a, b: a == b, ast.NotEq: lambda a, b: a != b,
    ast.Lt: lambda a, b: a < b, ast.LtE: lambda a, b: a <= b,
    ast.Gt: lambda a, b: a > b, ast.GtE: lambda a, b: a >= b,
}

import keyword

DEFAULT_DECLARATIONS = [
    {"name": "x", "type": "UInt", "width": 3},
    {"name": "y", "type": "UInt", "width": 3},
    {"name": "p", "type": "Bit", "width": 1},
    {"name": "q", "type": "Bit", "width": 1},
]

def validate_declarations(declarations):
    if len(declarations) > 32:
        raise ValueError("Use at most 32 variables in this demo.")
    seen = set()
    for row in declarations:
        name = row["name"]
        if not isinstance(name, str) or not name.isidentifier() or keyword.iskeyword(name):
            raise ValueError(f"{name!r} is not a valid variable name. Use a Python identifier, not a keyword.")
        if name in seen:
            raise ValueError(f"Variable {name!r} is declared twice. Names must be unique.")
        seen.add(name)
        if row["type"] not in ("UInt", "Bit"):
            raise ValueError(f"Unknown variable type for {name!r}.")
        if type(row["width"]) is not int or not 1 <= row["width"] <= 32:
            raise ValueError(f"Width of {name!r} must be between 1 and 32.")
        if row["type"] == "Bit" and row["width"] != 1:
            raise ValueError(f"Boolean variable {name!r} must have width 1.")
    return declarations

def parse_expression(source, width=3, declarations=None):
    if not source.strip():
        raise ValueError("Type an expression to begin.")
    if len(source) > 500:
        raise ValueError("Keep this live demo below 500 characters.")
    tree = ast.parse(source, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 150:
        raise ValueError("Expression is too large for the live demo.")
    if declarations is None:
        declarations = [dict(row, width=width if row["type"] == "UInt" else 1)
                        for row in DEFAULT_DECLARATIONS]
    validate_declarations(declarations)
    qlang.ASTNode.reset()
    names = {row["name"]: (qlang.uint(row["name"], row["width"]) if row["type"] == "UInt"
                           else qlang.var_bit(row["name"])) for row in declarations}

    def build(node):
        if isinstance(node, ast.Name):
            if node.id not in names:
                raise ValueError(f"Unknown variable {node.id!r}. Add it to the variable table on the right.")
            return names[node.id]
        if isinstance(node, ast.Constant):
            if type(node.value) not in (int, bool) or not 0 <= node.value <= 255:
                raise ValueError("Constants must be integers from 0 to 255 or booleans.")
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in BINARY:
            left, right = build(node.left), build(node.right)
            # Bound constant arithmetic as well as symbolic widths.
            if isinstance(node.op, (ast.LShift, ast.RShift)):
                if type(right) is not int or not 0 <= right <= 8:
                    raise ValueError("Shifts need a literal integer amount from 0 to 8.")
            return BINARY[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.Invert):
            value = build(node.operand)
            if not isinstance(value, qlang.Bit):
                raise ValueError("Use ~ only on Boolean expressions (for example ~p).")
            return ~value
        if isinstance(node, ast.Compare):
            if len(node.ops) != 1 or type(node.ops[0]) not in COMPARE:
                raise ValueError("Use separate comparisons joined by &: (x < y) & (y < 4).")
            return COMPARE[type(node.ops[0])](build(node.left), build(node.comparators[0]))
        raise ValueError("Use arithmetic, bitwise logic, and single comparisons; no calls or Python and/or/not.")

    root = build(tree.body)
    if type(root) is bool:
        root = qlang.const_bit(root)
    elif type(root) is int:
        if not 0 <= root <= 65535:
            raise ValueError("Constant result is outside the demo range.")
        root = qlang.uint(root)
    if not isinstance(root, qlang.ASTNode):
        raise ValueError("Expected a symbolic expression.")
    return root

# Pure SVG rendering: no Graphviz executable, JavaScript, or external assets.
SYMBOLS = {"ADD": "+", "SUB": "−", "MUL": "×", "LSHIFT": "<<", "RSHIFT": ">>",
           "AND": "&", "OR": "|", "XOR": "^", "NOT": "~", "EQ": "==",
           "NE": "!=", "LT": "<", "LE": "<=", "GT": ">", "GE": ">="}

def graph_data(root):
    nodes, edges, seen = [], [], set()
    def visit(node):
        key = id(node)
        if key in seen:
            return
        seen.add(key)
        nodes.append(node)
        for child in node.children:
            edges.append((key, id(child)))
            visit(child)
    visit(root)
    return nodes, edges

def node_text(node):
    kind = node.bit_type if isinstance(node, qlang.Bit) else node.uint_type
    if kind.name == "VAR":
        title, category = node.name, "variable"
    elif kind.name == "CONST":
        title = str(int(node.value)) if isinstance(node, qlang.Bit) else str(node.const_value)
        category = "constant"
    elif isinstance(node, qlang.Bit) and kind == BitType.UINT:
        title, category = f"bit {node.index}", "operation"
    else:
        title, category = SYMBOLS.get(kind.name, kind.name), "operation"
    subtitle = "Boolean" if isinstance(node, qlang.Bit) else f"UInt · {node.width} bits"
    return title, subtitle, category

def expression_text(root):
    """Readable, unambiguous expression represented by a semantic AST node."""
    cache = {}
    operators = dict(SYMBOLS, SUB="-", MUL="*")
    def format_node(node):
        key = id(node)
        if key in cache:
            return cache[key]
        kind = node.bit_type if isinstance(node, qlang.Bit) else node.uint_type
        if kind.name == "VAR":
            text = node.name
        elif kind.name == "CONST":
            text = str(int(node.value)) if isinstance(node, qlang.Bit) else str(node.const_value)
        elif isinstance(node, qlang.Bit) and kind == BitType.UINT:
            text = f"{format_node(node.parent_uint)}[{node.index}]"
        elif isinstance(node, qlang.UInt) and kind == UIntType.BITS:
            text = "bits(" + ", ".join(format_node(bit) for bit in node.bits) + ")"
        elif kind == BitType.NOT and isinstance(node, qlang.Bit):
            text = f"(~{format_node(node.operands[0])})"
        elif len(node.operands) == 2:
            left, right = (format_node(child) for child in node.operands)
            text = f"({left} {operators.get(kind.name, kind.name)} {right})"
        else:
            text = kind.name + "(" + ", ".join(format_node(child) for child in node.children) + ")"
        cache[key] = text
        return text
    result = format_node(root)
    return result[1:-1] if result.startswith("(") and result.endswith(")") else result


def abbreviated_label(text, limit=30):
    if len(text) <= limit:
        return text
    # Retain both ends; the complete expression is always in the SVG tooltip.
    left = (limit - 1) // 2
    return text[:left] + "…" + text[-(limit - 1 - left):]


def layout_graph(root, node_width=130):
    """Layered DAG layout with a reserved slot for every skipped-layer edge."""
    nodes, edges = graph_data(root)
    by_id = {id(n): n for n in nodes}
    depths = {id(root): 0}
    pending = [id(root)]
    while pending:
        parent = pending.pop()
        for child in by_id[parent].children:
            key = id(child)
            proposed = depths[parent] + 1
            if proposed > depths.get(key, -1):
                depths[key] = proposed
                pending.append(key)
    layers = defaultdict(list)
    for node in nodes:
        layers[depths[id(node)]].append(id(node))
    chains = []
    predecessors, successors = defaultdict(list), defaultdict(list)
    for index, (parent, child) in enumerate(edges):
        chain = [parent]
        for depth in range(depths[parent] + 1, depths[child]):
            dummy = ("route", index, depth)
            layers[depth].append(dummy)
            chain.append(dummy)
        chain.append(child)
        chains.append(chain)
        for a, b in zip(chain, chain[1:]):
            successors[a].append(b)
            predecessors[b].append(a)
    # Barycenter sweeps reduce edge crossings, while invisible routing slots
    # keep long connections clear of actual node boxes.
    for _ in range(4):
        for descending in (True, False):
            ranks = range(1, max(layers)+1) if descending else range(max(layers)-1, -1, -1)
            neighbors = predecessors if descending else successors
            for depth in ranks:
                order = {key: (i + .5) / len(layer)
                         for layer in layers.values() for i, key in enumerate(layer)}
                layers[depth].sort(key=lambda key: sum(order[n] for n in neighbors[key]) / len(neighbors[key])
                                   if neighbors[key] else order[key])
    width = max(640, max(map(len, layers.values())) * (node_width + 50) + 40)
    height = (max(layers) + 1) * 150 + 35
    positions = {key: ((i + .5) * width / len(layer), 45 + depth * 150)
                 for depth, layer in layers.items() for i, key in enumerate(layer)}
    outgoing_edges, incoming_edges = defaultdict(list), defaultdict(list)
    gap_segments = defaultdict(list)
    for index, (parent, child) in enumerate(edges):
        outgoing_edges[parent].append(index)
        incoming_edges[child].append(index)
        for a, b in zip(chains[index], chains[index][1:]):
            gap_segments[positions[a][1]].append((index, a, b))

    def port(index, indices):
        # Distinct attachment points: no shared vertical stems at either end.
        return 88 * ((indices.index(index) + .5) / len(indices) - .5)

    routes = []
    for index, chain in enumerate(chains):
        points = []
        for i, (a, b) in enumerate(zip(chain, chain[1:])):
            x1, y1 = positions[a]
            x2, y2 = positions[b]
            if i == 0:
                x1 += port(index, outgoing_edges[a])
            if i == len(chain)-2:
                x2 += port(index, incoming_edges[b])
            start_y = y1 + 28 if i == 0 else y1
            end_y = y2 - 28 if i == len(chain)-2 else y2
            segments = gap_segments[y1]
            lane = segments.index((index, a, b))
            # Every connection has its own horizontal track in this gap.
            mid_y = y1 + 51 + 48 * (lane + .5) / len(segments)
            if not points:
                points.append((x1, start_y))
            points.extend([(x1, mid_y), (x2, mid_y), (x2, end_y)])
        routes.append(points)
    return nodes, edges, positions, routes, width, height


def render_graph(root, full_expressions=False):
    node_width = 240 if full_expressions else 130
    nodes, edges, positions, routes, width, height = layout_graph(root, node_width=node_width)
    incoming = defaultdict(int)
    for _, child in edges:
        incoming[child] += 1
    palette = ["#2563eb", "#c2410c", "#15803d", "#9333ea", "#be185d", "#0e7490", "#a16207", "#475569"]
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="qlang expression graph" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="#f8f7f2"/>', '<defs>']
    for index in range(len(edges)):
        color = palette[index % len(palette)]
        parts.append(f'<marker id="arrow-{index}" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto"><path d="M 0 0 L 10 5 L 0 10 Z" fill="{color}"/></marker>')
    parts.append('</defs>')
    child_index = defaultdict(int)
    edge_labels = []
    by_id = {id(node): node for node in nodes}
    for index, ((parent, child), points) in enumerate(zip(edges, routes)):
        operand = child_index[parent]
        child_index[parent] += 1
        role = ("L" if operand == 0 else "R") if len(by_id[parent].children) == 2 else str(operand + 1)
        color = palette[index % len(palette)]
        description = html.escape(f'{node_text(by_id[parent])[0]}: operand {operand+1} → {node_text(by_id[child])[0]}')
        path = "M" + " L".join(f"{x},{y}" for x, y in points)
        # A background halo makes crossings look like crossings, not junctions.
        parts.append(f'<g><title>{description}</title><path d="{path}" fill="none" stroke="#f8f7f2" stroke-width="7" stroke-linejoin="round"/>')
        parts.append(f'<path d="{path}" fill="none" stroke="{color}" stroke-width="2.5" stroke-linejoin="round" marker-end="url(#arrow-{index})"/></g>')
        x, y = points[0]
        edge_labels.append(f'<g><title>{description}</title><circle cx="{x}" cy="{y+11}" r="9" fill="#f8f7f2" stroke="{color}"/><text x="{x}" y="{y+15}" text-anchor="middle" font-family="system-ui,sans-serif" font-size="10" font-weight="700" fill="{color}">{role}</text></g>')
    parts.extend(edge_labels)
    colors = {"variable": ("#dbeafe", "#1d4ed8"), "constant": ("#e2e8f0", "#334155"),
              "operation": ("#ede9fe", "#6d28d9")}
    for node in nodes:
        key = id(node)
        x, y = positions[key]
        title, subtitle, category = node_text(node)
        complete = expression_text(node)
        if full_expressions:
            title = abbreviated_label(complete)
        fill, stroke = colors[category]
        shared = incoming[key] > 1
        tooltip = html.escape(f"{complete}: {subtitle}" + (f" · shared by {incoming[key]} edges" if shared else ""))
        parts.append(f'<g><title>{tooltip}</title><rect x="{x-node_width/2}" y="{y-28}" width="{node_width}" height="56" rx="12" fill="{fill}" stroke="{stroke}" stroke-width="{3 if shared else 1}"/>')
        font_size = 12 if full_expressions else 18
        font_family = "ui-monospace,monospace" if full_expressions else "system-ui,sans-serif"
        parts.append(f'<text x="{x}" y="{y-3}" text-anchor="middle" font-family="{font_family}" font-size="{font_size}" font-weight="650" fill="{stroke}">{html.escape(str(title))}</text>')
        parts.append(f'<text x="{x}" y="{y+17}" text-anchor="middle" font-family="system-ui,sans-serif" font-size="11" fill="#475569">{html.escape(subtitle)}</text></g>')
    parts.append('</svg>')
    markup = '<style>.qlang-explorer, .qlang-explorer div, .qlang-explorer input, .qlang-explorer select, .qlang-explorer button {background-color:#f8f7f2;}</style><p style="color:#475569;font-size:12px">Each colored arrow connects one parent to one child. L = left operand; R = right operand. Crossings are not junctions.</p><div style="overflow:auto;max-height:760px;border:1px solid #e2e8f0;border-radius:12px">' + ''.join(parts) + '</div>'
    return markup, len(nodes), len(edges), sum(v > 1 for v in incoming.values())

def make_explorer():
    refresh_hook = {"callback": None}
    expression = widgets.Text(value="(x + y == 7) & (x > y)", placeholder="Type an expression…",
                              continuous_update=True, layout=widgets.Layout(width="100%"))
    optimized = widgets.Checkbox(value=False, description="Show optimized graph", indent=False)
    full_labels = widgets.Checkbox(value=True, description="Full expressions", indent=False,
                                  tooltip="Switch between operator symbols and complete expressions. Hover abbreviated labels to read the full expression.")
    presets = widgets.Dropdown(description="Examples", options=[
        ("Arithmetic constraint", "(x + y == 7) & (x > y)"),
        ("Shared subexpression", "((x + y) == 7) | ((x + y) == 5)"),
        ("Simplification", "((x + 0) == y) & (p | ~p)"),
        ("Boolean logic", "(p & q) | (~p & ~q)"),
        ("Unsigned subtraction", "(x - y) >= 2"),
    ], layout=widgets.Layout(width="100%"))
    rows = []
    changing_table = False
    table_body = widgets.VBox(layout=widgets.Layout(gap="6px"))
    table_message = widgets.HTML()
    status = widgets.HTML()
    graph = widgets.HTML()
    def refresh(change=None):
        if refresh_hook["callback"] is not None:
            return refresh_hook["callback"](change)
        started = time.perf_counter()
        try:
            declarations = [dict(name=row["name"].value, type=row["type"].value,
                                 width=row["width"].value) for row in rows]
            root = parse_expression(expression.value, declarations=declarations)
            if optimized.value:
                root = qlang.optimize_expr(root)
            svg, n, e, shared = render_graph(root, full_expressions=full_labels.value)
            graph.value = svg
            elapsed = (time.perf_counter() - started) * 1000
            status.value = (f'<p style="color:#475569">{n} nodes · {e} edges · {shared} shared nodes · '
                            f'{elapsed:.1f} ms parse/render · ' + ('optimized' if optimized.value else 'original') + '</p>')
        except Exception as exc:
            message = f"Incomplete expression: {exc.msg}" if isinstance(exc, SyntaxError) else str(exc)
            status.value = '<p style="color:#b45309">' + html.escape(message) + ' · Showing the last valid graph.</p>'
    def table_changed(change=None):
        if not changing_table:
            refresh()

    def add_row(name, kind="UInt", bits=3):
        name_field = widgets.Text(value=name, placeholder="name", continuous_update=True,
                                  layout=widgets.Layout(width="110px"))
        kind_field = widgets.Dropdown(options=[("UInt", "UInt"), ("Boolean", "Bit")], value=kind,
                                     layout=widgets.Layout(width="95px"))
        bit_field = widgets.BoundedIntText(value=bits, min=1, max=32, disabled=kind == "Bit",
                                          layout=widgets.Layout(width="65px"))
        remove = widgets.Button(description="×", tooltip="Remove variable",
                                layout=widgets.Layout(width="34px"))
        row = {"name": name_field, "type": kind_field, "width": bit_field}
        view = widgets.HBox([name_field, kind_field, bit_field, remove])
        row["view"] = view
        rows.append(row)
        table_body.children = tuple(r["view"] for r in rows)
        def type_changed(change):
            # Suspend refresh until type and width form one consistent declaration.
            nonlocal changing_table
            changing_table = True
            try:
                bit_field.disabled = change["new"] == "Bit"
                if bit_field.disabled:
                    bit_field.value = 1
            finally:
                changing_table = False
            refresh()
        def remove_row(_):
            rows.remove(row)
            table_body.children = tuple(r["view"] for r in rows)
            for control in (name_field, kind_field, bit_field, remove, view):
                control.close()
            refresh()
        name_field.observe(table_changed, names="value")
        bit_field.observe(table_changed, names="value")
        kind_field.observe(type_changed, names="value")
        remove.on_click(remove_row)

    def reset_variables(_=None):
        nonlocal changing_table
        changing_table = True
        try:
            for row in rows:
                for control in row.values():
                    control.close()
            rows.clear()
            for declaration in DEFAULT_DECLARATIONS:
                add_row(declaration["name"], declaration["type"], declaration["width"])
            table_message.value = ""
        finally:
            changing_table = False
        refresh()

    def add_variable(_):
        if len(rows) >= 32:
            table_message.value = "At most 32 variables are supported."
            return
        existing = {row["name"].value for row in rows}
        index = 1
        while f"v{index}" in existing:
            index += 1
        add_row(f"v{index}")
        table_message.value = ""
        refresh()

    add_button = widgets.Button(description="+ Add variable", layout=widgets.Layout(width="150px"))
    reset_button = widgets.Button(description="Reset defaults", layout=widgets.Layout(width="150px"))
    add_button.on_click(add_variable)
    reset_button.on_click(reset_variables)
    reset_variables()
    def choose(change):
        # Presets reference x/y/p/q; restore their declarations when selected.
        reset_variables()
        expression.value = change["new"]
    for control in (expression, optimized, full_labels):
        control.observe(refresh, names="value")
    presets.observe(choose, names="value")
    connection = widgets.Button(description="Test widget connection", layout=widgets.Layout(width="210px"))
    connection_status = widgets.HTML(value="Click the button to confirm frontend ↔ kernel communication.")
    def confirm_connection(_):
        connection_status.value = "<b style='color:#15803d'>Connected: browser events reach Python.</b>"
    connection.on_click(confirm_connection)
    refresh()
    headers = widgets.HBox([
        widgets.HTML("<b>Name</b>", layout=widgets.Layout(width="110px")),
        widgets.HTML("<b>Type</b>", layout=widgets.Layout(width="95px")),
        widgets.HTML("<b>Bits</b>", layout=widgets.Layout(width="65px")),
    ])
    variable_panel = widgets.VBox([
        widgets.HTML("<h3 style='margin:0'>Variables</h3><p>Edit names, types, and individual widths.</p>"),
        headers, table_body, widgets.HBox([add_button, reset_button]), table_message,
        widgets.HTML("<small>Names must be unique Python identifiers. Renaming does not rewrite the expression. "
                     "Selecting an example restores x, y, p, q.</small>"),
    ], layout=widgets.Layout(width="350px", min_width="350px", padding="12px", border="1px solid #cbd5e1"))
    canvas = widgets.VBox([presets, expression, widgets.HBox([optimized, full_labels]), status, graph, connection, connection_status],
                         layout=widgets.Layout(flex="1 1 380px", min_width="300px"))
    view = widgets.HBox([canvas, variable_panel],
                        layout=widgets.Layout(width="100%", align_items="flex-start", gap="16px", flex_flow="row wrap")).add_class("qlang-explorer")
    view.demo_controls = {"expression": expression, "optimized": optimized, "full_labels": full_labels,
                          "status": status, "graph": graph, "table": table_body,
                          "panel": variable_panel, "rows": rows, "presets": presets, "refresh_hook": refresh_hook,
                          "connection": connection, "connection_status": connection_status}
    return view




## Quantum schematic

The same expression compiles into a phase oracle. The circuit view retains named input/workspace registers, operation boxes, neutral box outlines, resource counts, validated domain summaries, and the previous valid circuit on errors. Toggle **Show operation boxes** to inspect the gates.

Only Boolean predicates can compile. Arithmetic-only expressions still display as ASTs and show a circuit validation error. Compilation support is narrower than classical evaluation. Domains describe intended search values; they do not constrain the oracle or run a solver. Large widths and complex expressions can produce expensive circuits.


In [4]:
# 16px line height + 16px padding/border allowance = 32px for one line.
expression_input = widgets.Text(
    value="x + y == 5",
    placeholder="(x + y == 5) & (flag | ~flag)",
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="32px", overflow="hidden"),
)
# 3 × 16px line height + 16px padding/border allowance = 64px.
declarations_input = widgets.Textarea(
    value="x = 3\ny = 3",
    placeholder="x = 3\nflag = bit",
    rows=3,
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="64px", overflow="hidden"),
)
domains_input = widgets.Textarea(
    value="x = 0..7\ny = 0..7",
    placeholder="x = 0..7\ny = 0, 2, 5",
    rows=3,
    continuous_update=True,
    layout=widgets.Layout(width="100%", height="64px", overflow="hidden"),
)
for control in (expression_input, declarations_input, domains_input):
    control.add_class("qlang-input")

field_label = lambda text: widgets.HTML(f"<strong>{text}</strong>")
expression_field = widgets.VBox([field_label("Expression"), expression_input], layout=widgets.Layout(width="100%", min_width="0", overflow="hidden"))
declarations_field = widgets.VBox([field_label("Variables"), declarations_input], layout=widgets.Layout(width="49%", min_width="0", overflow="hidden"))
domains_field = widgets.VBox([field_label("Domains"), domains_input], layout=widgets.Layout(width="49%", min_width="0", overflow="hidden"))
configuration_fields = widgets.HBox(
    [declarations_field, domains_field],
    layout=widgets.Layout(width="100%", min_width="0", justify_content="space-between", overflow="hidden"),
)
for container in (expression_field, declarations_field, domains_field, configuration_fields):
    container.add_class("qlang-input-wrapper")

status = widgets.HTML("<em>Waiting for the first compilation.</em>")
show_operation_boxes = widgets.Checkbox(value=True, description="Show operation boxes")
circuit_mode = widgets.Dropdown(
    options=[("Forward compute", "compute"), ("Phase oracle", "oracle")],
    value="compute",
    description="Circuit:",
)
circuit_controls = widgets.HBox([show_operation_boxes, circuit_mode])
oracle_info_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_width="0", overflow="hidden", padding="0 8px")
)
circuit_figure_output = widgets.Output(
    layout=widgets.Layout(width="100%", min_width="0", overflow="auto", padding="8px")
)
with circuit_figure_output:
    display(HTML("<em>The compiled circuit will appear here.</em>"))

circuit_title = widgets.HTML("<h4 style='margin: 0 0 8px'>Forward reversible computation</h4>")
circuit_panel = widgets.VBox(
    [circuit_title, oracle_info_output, circuit_figure_output],
    layout=widgets.Layout(width="100%", min_width="0", border="1px solid #cbd5e1", padding="12px", overflow="hidden"),
)
circuit_panel.add_class("qlang-circuit-panel")

dashboard_styles = widgets.HTML(
    """<style>
    .qlang-dashboard, .qlang-dashboard .widget-box, .qlang-dashboard .widget-output {
        box-sizing: border-box;
        max-width: 100%;
    }
    .qlang-dashboard .qlang-input-wrapper,
    .qlang-dashboard .qlang-input,
    .qlang-dashboard .qlang-input textarea {
        box-sizing: border-box !important;
        min-width: 0 !important;
        max-width: 100% !important;
        width: 100% !important;
    }
    .qlang-dashboard .qlang-input { overflow-x: hidden !important; }
    .qlang-dashboard textarea.widget-input {
        overflow: hidden !important;
        overflow-wrap: anywhere;
        resize: none;
    }
    .qlang-heatmap-stale { opacity: 0.35; }
    .qlang-circuit-panel { transition: opacity 160ms ease; }
    .qlang-circuit-panel.qlang-dimmed { opacity: 0.35; }
    </style>"""
)
dashboard = widgets.VBox(
    [dashboard_styles, expression_field, configuration_fields, status, show_operation_boxes, circuit_panel],
    layout=widgets.Layout(width="90%", max_width="90%", min_width="0", margin="0 auto", grid_gap="10px", overflow="hidden"),
)
dashboard.add_class("qlang-dashboard")



In [5]:
from html import escape
from keyword import iskeyword

_LINE = re.compile(r"^\s*([A-Za-z_]\w*)\s*=\s*(.*?)\s*$")
_RANGE = re.compile(r"^(\d+)\s*\.\.\s*(\d+)$")
_LIST = re.compile(r"^\d+(?:\s*,\s*\d+)*$")
_BINARY_OPERATORS = {ast.Add, ast.Sub, ast.Mult, ast.LShift, ast.RShift, ast.BitAnd, ast.BitOr, ast.BitXor}
_COMPARISON_OPERATORS = {ast.Eq, ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE}


def _lines(text, label):
    parsed = {}
    for line_number, raw_line in enumerate(text.splitlines(), start=1):
        if not raw_line.strip():
            continue
        match = _LINE.fullmatch(raw_line)
        if not match:
            raise ValueError(f"{label} line {line_number} must look like 'name = value'")
        name, value = match.groups()
        if iskeyword(name):
            raise ValueError(f"{label} line {line_number} uses Python keyword '{name}'")
        if name in parsed:
            raise ValueError(f"{label} declares '{name}' more than once")
        parsed[name] = value
    if not parsed:
        raise ValueError(f"Provide at least one {label.lower()} entry")
    return parsed




def _domains(text, widths):
    declarations = _lines(text, "Domain")
    missing = set(widths) - set(declarations)
    unexpected = set(declarations) - set(widths)
    if missing:
        raise ValueError(f"Missing domains for: {', '.join(sorted(missing))}")
    if unexpected:
        raise ValueError(f"Unknown domain variables: {', '.join(sorted(unexpected))}")

    domains = {}
    for name, declaration in declarations.items():
        range_match = _RANGE.fullmatch(declaration)
        if range_match:
            start, stop = map(int, range_match.groups())
            if start > stop:
                raise ValueError(f"Domain '{name}' has a descending range")
            if stop - start + 1 > 4096:
                raise ValueError("Keep a live domain below 4097 values")
            values = tuple(range(start, stop + 1))
        elif _LIST.fullmatch(declaration):
            values = tuple(sorted({int(value.strip()) for value in declaration.split(",")}))
        else:
            raise ValueError(f"Domain '{name}' needs 'start..stop' or comma-separated integers")
        limit = 1 << widths[name]
        if not values or values[0] < 0 or values[-1] >= limit:
            raise ValueError(f"Domain '{name}' must fit its {widths[name]}-bit declaration (0..{limit - 1})")
        domains[name] = values
    return domains






def _labeled_display_circuit(oracle, show_boxes):
    """Build a display-only copy with named input registers and boxed gadgets."""
    source = oracle.circuit
    output_indices = tuple(getattr(oracle, "outputs", ()))
    input_runs, input_bits = {}, {}
    subscript_digits = str.maketrans("0123456789", "₀₁₂₃₄₅₆₇₈₉")
    for name, indices in oracle.inputs.items():
        for bit_index, qubit_index in enumerate(indices):
            input_bits[qubit_index] = name if len(indices) == 1 else f"{name}_{bit_index}"
        if not set(indices) & set(output_indices) and tuple(indices) == tuple(range(indices[0], indices[0] + len(indices))):
            input_runs[indices[0]] = (name, indices)

    registers, index = [], 0
    ancilla_number = 0
    while index < source.num_qubits:
        if index in output_indices:
            output_number = output_indices.index(index)
            output_name = "out" if len(output_indices) == 1 else f"out_{output_number}"
            registers.append(QuantumRegister(1, output_name))
            index += 1
            continue
        run = input_runs.get(index)
        if run is not None:
            name, indices = run
            registers.append(QuantumRegister(len(indices), name))
            index += len(indices)
            continue
        input_name = input_bits.get(index)
        if input_name is not None:
            registers.append(QuantumRegister(1, input_name))
            index += 1
            continue
        registers.append(QuantumRegister(1, f"anc{str(ancilla_number).translate(subscript_digits)}"))
        ancilla_number += 1
        index += 1

    display_circuit = QuantumCircuit(*registers)

    def append_instruction(instruction):
        qubits = [display_circuit.qubits[source.find_bit(qubit).index] for qubit in instruction.qubits]
        display_circuit.append(instruction.operation, qubits, instruction.clbits)

    def append_box(start, end, label):
        instructions = source.data[start:end]
        touched = sorted({source.find_bit(qubit).index for instruction in instructions for qubit in instruction.qubits})
        if not touched:
            return
        local_index = {qubit_index: local for local, qubit_index in enumerate(touched)}
        body = QuantumCircuit(len(touched))
        for instruction in instructions:
            body.append(
                instruction.operation,
                [body.qubits[local_index[source.find_bit(qubit).index]] for qubit in instruction.qubits],
            )
        display_circuit.box(body, qubits=[display_circuit.qubits[qubit_index] for qubit_index in touched], clbits=[], label=label)

    position, box_labels = 0, []
    if show_boxes:
        for number, (label, start, end) in enumerate(getattr(oracle, "operation_spans", []), start=1):
            for instruction in source.data[position:start]:
                append_instruction(instruction)
            box_label = f"{number}. {label}"
            append_box(start, end, box_label)
            box_labels.append(box_label)
            position = end
    for instruction in source.data[position:]:
        append_instruction(instruction)
    return display_circuit, box_labels


def _set_error(message):
    expression_input.layout.border = "2px solid #dc2626"
    circuit_panel.add_class("qlang-dimmed")
    status.value = f"<span style='color: #b91c1c'><strong>Compilation error:</strong> {escape(str(message))}</span>"


def _set_compiling():
    expression_input.layout.border = ""
    circuit_panel.remove_class("qlang-dimmed")
    status.value = "<span style='color: #1d4ed8'><strong>Compiling…</strong></span>"


_OPERATION_BOX_COLOR = "#6b7280"


def _style_operation_boxes(figure, labels):
    """Style BoxOp outlines and draw their labels explicitly."""
    label_iter = iter(labels)
    for axis in figure.axes:
        for patch in axis.patches:
            if isinstance(patch, FancyBboxPatch):
                patch.set_edgecolor(_OPERATION_BOX_COLOR)
                patch.set_linestyle(":")
                label = next(label_iter, None)
                if label is not None:
                    bounds = patch.get_bbox().bounds
                    top = bounds[1] if axis.yaxis_inverted() else bounds[1] + bounds[3]
                    upward_offset = -0.16 if axis.yaxis_inverted() else 0.16
                    axis.text(bounds[0], top + upward_offset, label, ha="left", va="bottom", fontsize=11, color=_OPERATION_BOX_COLOR, zorder=patch.get_zorder() + 1, clip_on=False)


_last_successful = None


def _set_success(compiled, domains, *, remember=True):
    global _last_successful
    expression_input.layout.border = ""
    circuit_panel.remove_class("qlang-dimmed")
    domain_summary = "; ".join(f"{name}: {list(values)}" for name, values in domains.items())
    display_circuit, box_labels = _labeled_display_circuit(compiled, show_operation_boxes.value)
    box_summary = " · ".join(box_labels) if box_labels else "none"
    ancilla_count = len(compiled.clean_ancillas) + len(compiled.dirty_ancillas)
    output_summary = "phase-marked predicate" if circuit_mode.value == "oracle" else str(compiled.outputs)
    circuit_title.value = "<h4 style='margin: 0 0 8px'>Compiled phase oracle</h4>" if circuit_mode.value == "oracle" else "<h4 style='margin: 0 0 8px'>Forward reversible computation</h4>"
    oracle_info = (
        f"inputs: {compiled.inputs}\n"
        f"outputs: {output_summary}\n"
        f"qubits: {compiled.circuit.num_qubits}\n"
        f"ancillas: {ancilla_count}\n"
        f"gates: {compiled.circuit.size()}\n"
        f"validated domains: {domain_summary}\n"
        f"operation boxes: {box_summary}"
    )
    # Render before clearing the existing output, so the previous circuit never flashes blank.
    figure = display_circuit.draw("mpl", fold=-1)
    if box_labels:
        _style_operation_boxes(figure, box_labels)
    try:
        with oracle_info_output:
            clear_output(wait=True)
            display(HTML(
                "<pre style='font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, monospace; "
                "margin: 0 0 8px; white-space: pre-wrap'>"
                f"{escape(oracle_info)}</pre>"
            ))
        with circuit_figure_output:
            clear_output(wait=True)
            display(figure)
    finally:
        plt.close(figure)
    if remember:
        _last_successful = (compiled, domains)
    status.value = "<span style='color: #166534'><strong>Compiled successfully.</strong> Domains validated; no search was run.</span>"


_refreshing = False


def _rerender_operation_boxes(_change=None):
    if _last_successful is not None:
        previous_status, previous_border = status.value, expression_input.layout.border
        stale = "Compilation error:" in previous_status
        _set_success(*_last_successful, remember=False)
        if stale:
            status.value = previous_status
            expression_input.layout.border = previous_border
            circuit_panel.add_class("qlang-dimmed")






## Solution heatmap and Grover probabilities

The third adapter reads the shared controller's expression, variable widths, domains, and compiled oracle—there is no new parser or second oracle compiler. Choose **Satisfying assignments**, **Ideal Grover probability**, or **Sampled Grover frequency**. Move **Grover k** to see ideal amplification immediately; click **Run Grover** to sample the same oracle through qlang's Grover circuit builder. Changing only axes, iterations, or shots does not rebuild the AST or oracle.

Columns and rows select oracle input variables. Other variables are summed over their allowed domains. Probabilities remain globally normalized over the full input register: domain filtering may make the visible mass less than 100%. Variables removed by optimization do not appear as axes. Purple cell borders indicate at least one satisfying completion. Probability color scales stay fixed across iteration changes for the same axes/domains; labels show exact predicted probabilities or sampled frequencies.

Live enumeration is capped at 8 input bits. On-demand statevector simulation is capped at 18 total qubits, including workspace, and 12,000 circuit gates. These limits apply only to the heatmap; the AST and schematic continue working. Predictions assume ideal amplitude amplification and do not verify compiler correctness. [Grover amplitude-amplification reference](https://qiskit-community.github.io/qiskit-algorithms/tutorials/06_grover.html).


In [6]:
"""Heatmap adapter for the same predicate, schema, and oracle as the other views."""
from dataclasses import dataclass
from itertools import product
from math import asin, sin, sqrt


MAX_HEATMAP_INPUT_BITS = 8
MAX_HEATMAP_SIM_QUBITS = 18
MAX_HEATMAP_SIM_GATES = 12000


@dataclass
class HeatmapInputs:
    names: tuple
    assignments: tuple
    feasible: tuple


def enumerate_heatmap_inputs(predicate, oracle, widths):
    """Read the real oracle register layout; never parse another expression."""
    names = tuple(oracle.inputs)
    bit_count = sum(len(oracle.inputs[name]) for name in names)
    if bit_count > MAX_HEATMAP_INPUT_BITS:
        raise ValueError(f"Live heatmaps support at most {MAX_HEATMAP_INPUT_BITS} input bits; this oracle has {bit_count}. Reduce widths to enable the heatmap. AST and circuit remain available.")
    if not names:
        raise ValueError("The simplified oracle has no input registers to plot.")
    assignments = tuple(product(*(range(1 << len(oracle.inputs[name])) for name in names)))
    defaults = dict.fromkeys(widths, 0)
    feasible = tuple(bool(qlang.evaluate(predicate, {**defaults, **dict(zip(names, row))}))
                     for row in assignments)
    return HeatmapInputs(names, assignments, feasible)


def ideal_grover_probabilities(feasible, iterations):
    """Uniform initial state and an ideal clean phase oracle; not simulation."""
    total, marked = len(feasible), sum(feasible)
    if not total:
        return ()
    if marked in (0, total):
        return (1 / total,) * total
    success = sin((2 * iterations + 1) * asin(sqrt(marked / total))) ** 2
    return tuple(success / marked if good else (1 - success) / (total - marked) for good in feasible)


def sample_grover_distribution(oracle, iterations, shots):
    """Execute qlang's existing Grover circuit builder with this exact oracle."""
    from qiskit.quantum_info import Statevector
    from qlang.qalgorithm.grover import _grover_circuit

    if oracle.circuit.num_qubits > MAX_HEATMAP_SIM_QUBITS:
        raise ValueError(f"Simulation needs {oracle.circuit.num_qubits} total qubits, including workspace; the interactive limit is {MAX_HEATMAP_SIM_QUBITS}.")
    circuit, inputs = _grover_circuit(oracle, k=iterations)
    if circuit.size() > MAX_HEATMAP_SIM_GATES:
        raise ValueError("This Grover circuit exceeds the interactive gate budget. Reduce the iteration count or expression size.")
    state = Statevector.from_instruction(circuit)
    state.seed(7)
    counts = state.sample_counts(shots, qargs=inputs)
    # _grover_circuit uses oracle.inputs order, with each register LSB first.
    distribution = {}
    for bitstring, count in counts.items():
        value, offset, row = int(bitstring, 2), 0, []
        for qubits in oracle.inputs.values():
            row.append((value >> offset) & ((1 << len(qubits)) - 1))
            offset += len(qubits)
        distribution[tuple(row)] = int(count) / shots
    return distribution


def project_heatmap(data, weights, domains, x_name, y_name):
    """Marginalize other variables over allowed domains without renormalizing."""
    columns = list(domains[x_name])
    rows = list(domains[y_name]) if y_name else [None]
    xi, yi = data.names.index(x_name), data.names.index(y_name) if y_name else None
    allowed = {name: set(domains[name]) for name in data.names}
    cells = {(x, y): 0.0 for y in rows for x in columns}
    marked = dict.fromkeys(cells, 0)
    for assignment, good, weight in zip(data.assignments, data.feasible, weights):
        if any(value not in allowed[name] for name, value in zip(data.names, assignment)):
            continue
        key = assignment[xi], assignment[yi] if yi is not None else None
        cells[key] += weight
        marked[key] += int(good)
    return columns, rows, cells, marked


def render_solution_heatmap(data, weights, domains, x_name, y_name, probability):
    columns, rows, values, marked = project_heatmap(data, weights, domains, x_name, y_name)
    size, left, top = 46, 78, 55
    width, height = left + size * len(columns) + 20, top + size * len(rows) + 30
    maximum = max(values.values(), default=0) or 1
    if probability:
        good, total = sum(data.feasible), len(data.feasible)
        good_weights = tuple((1 / good if marked else 0) if good else 0 for marked in data.feasible)
        bad_weights = tuple((0 if marked else 1 / (total-good)) if good < total else 0 for marked in data.feasible)
        good_cells = project_heatmap(data,good_weights,domains,x_name,y_name)[2]
        bad_cells = project_heatmap(data,bad_weights,domains,x_name,y_name)[2]
        maximum = max([*good_cells.values(),*bad_cells.values()],default=1) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" role="img" aria-label="Solution heatmap" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
             '<rect width="100%" height="100%" fill="#f8f7f2"/>',
             f'<text x="{left}" y="18" font-family="system-ui" font-size="13">Columns: {escape(x_name)} · Rows: {escape(y_name or "all other variables")}</text>']
    for column, value in enumerate(columns):
        parts.append(f'<text x="{left+(column+.5)*size}" y="42" text-anchor="middle" font-family="system-ui" font-size="11">{value}</text>')
    for row, value in enumerate(rows):
        label = str(value) if value is not None else "total"
        parts.append(f'<text x="{left-12}" y="{top+(row+.55)*size}" text-anchor="end" font-family="system-ui" font-size="11">{label}</text>')
        for column, x in enumerate(columns):
            key = x, value
            weight = values[key]
            strength = min(1,weight / maximum)
            rgb = tuple(round(a+(b-a)*strength) for a,b in zip((248,247,242),(37,99,235)))
            fill = '#%02x%02x%02x' % rgb
            border = '#7c3aed' if marked[key] else '#d5d7db'
            label = f'{weight:.1%}' if probability else str(round(weight))
            description = escape(f'{x_name}={x}' + (f', {y_name}={value}' if y_name else '')
                                 + f'; {marked[key]} satisfying completion(s); displayed value {label}')
            px, py = left+column*size, top+row*size
            parts.append(f'<g><title>{description}</title><rect x="{px+2}" y="{py+2}" width="42" height="42" rx="4" fill="{fill}" stroke="{border}" stroke-width="{2 if marked[key] else 1}"/>')
            color = '#ffffff' if strength > .6 else '#202c3b'
            parts.append(f'<text x="{px+size/2}" y="{py+size*.58}" text-anchor="middle" font-family="system-ui" font-size="10" fill="{color}">{label}</text></g>')
    parts.append('</svg>')
    return '<div style="overflow:auto;max-height:620px">'+''.join(parts)+'</div>', sum(values.values())


class GroverHeatmap:
    """A renderer adapter: the controller owns parsing, domains, and compilation."""
    def __init__(self, model):
        self.model = model
        self.data = None
        self.domains = None
        self.oracle = None
        self.updating = False
        self.sampled = None
        self.sample_key = None
        self.task = None
        self.closed = False
        self.stale = True
        self.mode = widgets.Dropdown(options=[('Satisfying assignments','solutions'),('Ideal Grover probability','ideal'),('Sampled Grover frequency','sampled')],value='solutions',description='View',layout=widgets.Layout(width='340px'))
        self.x_axis = widgets.Dropdown(description='Columns',layout=widgets.Layout(width='230px'))
        self.y_axis = widgets.Dropdown(description='Rows',layout=widgets.Layout(width='230px'))
        self.iterations = widgets.IntSlider(value=1,min=0,max=8,description='Grover k',continuous_update=False)
        self.shots = widgets.Dropdown(options=[256,1024,4096],value=1024,description='Shots',layout=widgets.Layout(width='180px'))
        self.run = widgets.Button(description='Run Grover',button_style='primary',disabled=True)
        self.note = widgets.HTML()
        self.plot = widgets.HTML()
        self.panel = widgets.VBox([
            widgets.HTML('<h3>Solutions and Grover amplification</h3><p>Live solution counts and ideal predictions; simulation runs only when you click Run Grover.</p>'),
            widgets.HBox([self.mode,self.x_axis,self.y_axis],layout=widgets.Layout(flex_flow='row wrap')),
            widgets.HBox([self.iterations,self.shots,self.run],layout=widgets.Layout(flex_flow='row wrap')),
            self.note,self.plot,
            widgets.HTML('<small>Purple borders indicate at least one satisfying completion. Other input variables are summed over allowed domains. Domains filter the display, not the oracle. Probabilities are global and are not renormalized to the visible domain. Ideal predictions assume a correct oracle; sampled frequencies execute the compiled circuit. Live limit: 8 input bits; simulation limit: 18 total qubits, including workspace.</small>'),
        ],layout=widgets.Layout(border='1px solid #cbd5e1',padding='12px',width='100%'))
        for control in (self.mode,self.x_axis,self.y_axis,self.iterations,self.shots):
            control.observe(self.redraw,names='value')
        self.run.on_click(self.request_sample)

    def mark_stale(self, message):
        self.stale = True
        self.run.disabled = True
        self.note.value = '<p style="color:#b45309">'+escape(str(message))+' Previous heatmap is stale.</p>'
        self.plot.add_class('qlang-heatmap-stale')

    def update(self, domains):
        try:
            oracle = self.model.oracle
            if oracle is not self.oracle or self.data is None:
                self.data = enumerate_heatmap_inputs(self.model.predicate,oracle,self.model.widths)
                self.oracle = oracle
                self.sampled = self.sample_key = None
            self.domains = domains
            self.stale = False
            self.updating = True
            old_x, old_y = self.x_axis.value, self.y_axis.value
            self.x_axis.options = self.data.names
            self.x_axis.value = old_x if old_x in self.data.names else self.data.names[0]
            self.y_axis.options = [('Marginal total',None)]+[(name,name) for name in self.data.names]
            candidates = [name for name in self.data.names if name != self.x_axis.value]
            self.y_axis.value = old_y if old_y in candidates else (candidates[0] if candidates else None)
            self.plot.remove_class('qlang-heatmap-stale')
        except Exception as error:
            self.mark_stale(error)
        finally:
            self.updating = False
        self.redraw()

    def key(self):
        return id(self.oracle),self.iterations.value,self.shots.value

    def redraw(self,change=None):
        if self.updating or self.data is None or self.closed or self.stale:
            return
        if self.y_axis.value == self.x_axis.value:
            self.updating = True
            self.y_axis.value = next((name for name in self.data.names if name != self.x_axis.value),None)
            self.updating = False
        total,good = len(self.data.feasible),sum(self.data.feasible)
        predicted = ideal_grover_probabilities(self.data.feasible,self.iterations.value)
        success = sum(p for p,marked in zip(predicted,self.data.feasible) if marked)
        can_sample = self.oracle.circuit.num_qubits <= MAX_HEATMAP_SIM_QUBITS
        self.run.disabled = not can_sample or (self.task is not None and not self.task.done())
        mode = self.mode.value
        if mode=='solutions':
            weights = tuple(map(int,self.data.feasible))
            description = 'Classical satisfying-assignment counts'
        elif mode=='ideal':
            weights = predicted
            description = 'Analytical ideal Grover prediction (not simulation)'
        elif self.sampled is not None and self.sample_key==self.key():
            weights = tuple(self.sampled.get(row,0) for row in self.data.assignments)
            description = f'Compiled-circuit Grover simulation · {self.shots.value} shots'
        else:
            self.note.value = '<p>Select Run Grover to produce samples for the current iteration count and shots. No simulation runs while typing.</p>'
            self.plot.value = ''
            return
        markup,visible = render_solution_heatmap(self.data,weights,self.domains,self.x_axis.value,self.y_axis.value,mode!='solutions')
        self.plot.value = markup
        shown = f'{visible:.1%} probability mass displayed' if mode!='solutions' else f'{int(visible)} satisfying assignments within the domains'
        warning = '' if can_sample else f' · Simulation disabled: {self.oracle.circuit.num_qubits} total qubits exceeds {MAX_HEATMAP_SIM_QUBITS}'
        self.note.value = (f'<p><b>{description}</b><br>{good}/{total} satisfying states across the full oracle input register · '
                           f'Ideal success at k={self.iterations.value}: {success:.1%}<br>{shown}{warning}</p>')

    def request_sample(self,_=None):
        if self.data is None or self.closed or self.stale:
            return
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            self.sampled = sample_grover_distribution(self.oracle,self.iterations.value,self.shots.value)
            self.sample_key = self.key()
            self.mode.value = 'sampled'
            self.redraw()
        else:
            if self.task is not None and not self.task.done():
                return
            self.task = loop.create_task(self.sample_async())

    async def sample_async(self):
        key,oracle = self.key(),self.oracle
        self.run.disabled = True
        self.note.value = '<p>Simulating the compiled Grover circuit…</p>'
        try:
            sampled = await asyncio.to_thread(sample_grover_distribution,oracle,key[1],key[2])
            if not self.closed and self.data is not None and not self.stale and key==self.key():
                self.sampled,self.sample_key = sampled,key
                self.mode.value = 'sampled'
                self.redraw()
        except Exception as error:
            if not self.closed and key==self.key():
                self.note.value = '<p style="color:#b45309">'+escape(str(error))+'</p>'
        finally:
            if not self.closed:
                self.run.disabled = self.stale or self.data is None or self.oracle.circuit.num_qubits > MAX_HEATMAP_SIM_QUBITS

    def close(self):
        self.closed = True
        if self.task is not None:
            self.task.cancel()
        self.panel.close()


In [7]:
# Shared expression model: a single parser and schema feed both renderers.
import asyncio
if "merged_controller" in globals():
    merged_controller.close()

class MergedController:
    def __init__(self):
        global expression_input, _last_successful
        self.view = make_explorer()
        self.controls = self.view.demo_controls
        expression_input = self.controls["expression"]
        self.pending = None
        self.closed = False
        self.domain_sync = False
        self.signature = self.schema = self.predicate = self.oracle = self.forward = None
        self.widths = {}
        self.model_error = None
        self.heatmap = GroverHeatmap(self)
        _last_successful = None
        panel = self.controls["panel"]
        domains_field.layout.width = "100%"
        panel.children = (*panel.children, domains_field)
        self.controls["refresh_hook"]["callback"] = self.sync
        show_operation_boxes.observe(_rerender_operation_boxes,names="value")
        circuit_mode.observe(self.schedule,names="value")
        domains_input.observe(self.schedule,names="value")
        self.sync()
        self.flush()
        self.dashboard = widgets.VBox([
            dashboard_styles,self.view,
            widgets.HTML("<h3>Quantum schematic</h3><p>Both views share one parsed expression and variable schema. Domains are validated, not encoded into the oracle. No quantum simulation runs.</p>"),
            status,circuit_controls,circuit_panel,self.heatmap.panel,
        ],layout=widgets.Layout(width="100%",min_width="0",gap="12px"))
        self.dashboard.add_class("qlang-dashboard")
        self.dashboard.add_class("qlang-explorer")

    def sync(self,change=None):
        if self.closed: return
        declarations = [dict(name=r["name"].value,type=r["type"].value,width=r["width"].value) for r in self.controls["rows"]]
        try:
            validate_declarations(declarations)
            schema = tuple((r["name"],r["type"],r["width"]) for r in declarations)
            signature = (expression_input.value,schema)
            changed = signature != self.signature
            if changed:
                self.heatmap.mark_stale("Expression changed; awaiting the shared model.")
                self.predicate = parse_expression(expression_input.value,declarations=declarations)
                self.signature = signature
                self.oracle = self.forward = None
            self.model_error = None
            root = qlang.optimize_expr(self.predicate) if self.controls["optimized"].value else self.predicate
            markup,count,edges,shared = render_graph(root,full_expressions=self.controls["full_labels"].value)
            self.controls["graph"].value = markup
            self.controls["status"].value = f"<p>{count} nodes · {edges} edges · {shared} shared nodes · " + ("optimized" if self.controls["optimized"].value else "original") + "</p>"
            if schema != self.schema:
                try: old = _lines(domains_input.value,"Domain")
                except ValueError: old = {}
                self.widths = {r["name"]:r["width"] for r in declarations}
                rebuilt = []
                for row in declarations:
                    name,width = row["name"],row["width"]
                    value = old.get(name,f"0..{(1 << width)-1}")
                    try: _domains(f"{name} = {value}",{name:width})
                    except ValueError: value = f"0..{(1 << width)-1}"
                    rebuilt.append(f"{name} = {value}")
                self.domain_sync = True
                try: domains_input.value = "\n".join(rebuilt)
                finally: self.domain_sync = False
                self.schema = schema
            if changed: self.schedule()
        except Exception as error:
            self.cancel()
            self.model_error = str(error)
            self.controls["status"].value = "<p style='color:#b45309'>" + escape(str(error)) + " · Showing the last valid graph.</p>"
            _set_error(error)
            self.heatmap.mark_stale(error)

    def cancel(self):
        if self.pending is not None:
            self.pending.cancel()
            self.pending = None

    def schedule(self,change=None):
        if self.closed or self.domain_sync: return
        self.cancel()
        self.heatmap.mark_stale("Waiting for the current domains and oracle.")
        status.value = "<span style='color:#475569'>Waiting for typing to pause…</span>"
        try: loop = asyncio.get_running_loop()
        except RuntimeError: self.flush()
        else: self.pending = loop.call_later(0.35,self.flush)

    def flush(self):
        self.cancel()
        if self.closed: return
        try:
            if self.model_error is not None: raise ValueError(self.model_error)
            _set_compiling()
            domains = _domains(domains_input.value,self.widths)
            if not isinstance(self.predicate,qlang.Bit):
                raise TypeError("The expression must produce a Boolean predicate, such as 'x + y == 5'")
            if circuit_mode.value == "compute":
                if self.forward is None: self.forward = qlang.compile_circuit(self.predicate)
                compiled = self.forward
            else:
                if self.oracle is None: self.oracle = qlang.compile_oracle(self.predicate)
                compiled = self.oracle
            _set_success(compiled,domains)
            if self.oracle is None: self.oracle = qlang.compile_oracle(self.predicate)
            self.heatmap.update(domains)
        except Exception as error:
            _set_error(error)
            self.heatmap.mark_stale(error)

    def close(self):
        self.closed = True
        self.heatmap.close()
        self.cancel()
        self.controls["refresh_hook"]["callback"] = None
        domains_input.unobserve(self.schedule,names="value")
        show_operation_boxes.unobserve(_rerender_operation_boxes,names="value")
        circuit_mode.unobserve(self.schedule,names="value")
        self.view.close()
        if hasattr(self,"dashboard"): self.dashboard.close()

merged_controller = MergedController()
merged_dashboard = merged_controller.dashboard
display(merged_dashboard)
print("Merged AST + schematic ready. One parsed expression feeds both views.")


Merged AST + schematic ready. One parsed expression feeds both views.


## Raw qlang API

The live views above are useful for exploration. These small cells show the same workflow directly in Python: build an expression, inspect its optimization, evaluate it classically, and compile its forward circuit and phase oracle. The `raw_` names intentionally avoid the live editor's variables.


In [ ]:
raw_x = qlang.uint("raw_x", 3)
raw_y = qlang.uint("raw_y", 2)
raw_predicate = raw_x + raw_y == 5

print("Original AST:")
qlang.print_ast(raw_predicate)
print("\nOptimized AST:")
qlang.print_ast(qlang.optimize_expr(raw_predicate))
print("\nClassical check at raw_x=3, raw_y=2:", qlang.evaluate(raw_predicate, {"raw_x": 3, "raw_y": 2}))

raw_forward = qlang.compile_circuit(raw_predicate)
raw_oracle = qlang.compile_oracle(raw_predicate)
print("\nForward circuit:")
print(raw_forward.circuit.draw("text"))
print("Phase-oracle qubits:", raw_oracle.circuit.num_qubits)


In [ ]:
# Grover/BBHT search is also available directly from the same predicate.
raw_solution = qlang.search({"raw_x": range(8), "raw_y": range(4)}, raw_predicate)
print("Raw search result:", raw_solution)


## Demo sequence

1. Edit `(x + y == 7) & (x > y)` and inspect both views.
2. Switch **Full expressions** off/on, then **Show operation boxes** off/on.
3. Select **Simplification** and toggle the optimized AST.
4. Add a UInt named `budget` with width 3 and type `budget >= 2`; check its named circuit register and generated domain.
5. Set its domain to `budget = 0..8` to see validation, then correct it to `budget = 0..7`.
6. Enter incomplete input: both views retain their last valid visualization and explain the error.

To rerun safely, use **Run All** (setup imports follow the schematic notebook). This notebook compiles circuits; it does not execute Grover or QAOA.

7. In the heatmap, choose **Ideal Grover probability**, move **Grover k**, then click **Run Grover** to compare with sampled frequencies.
